#### Import the necessary modules and packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

#### Define the subset of features to use

In [ ]:
features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
    "internal_resistance_mOhm",
    "cumulative_high_temp_hours",
]
features_B = features_A + ["cycle_index"]
features_C = features_B + ["ambient_temp_c"]
features = [features_A, features_B, features_C] # all feature sets together

#### Check if any of the variables contain NaN-values

In [ ]:
import pandas as pd
import numpy as np

for feature in features:

    X_train_raw = train[feature]
    X_val_raw = val[feature]

    # 1. Total NaNs per feature
    print("Train NaNs per column:")
    print(X_train_raw.isna().sum().sort_values(ascending=False))

    print("\nValidation NaNs per column:")
    print(X_val_raw.isna().sum().sort_values(ascending=False))

    # 2. Overall counts
    print("\nTrain rows with any NaN:", X_train_raw.isna().any(axis=1).sum())
    print("Train total NaN cells:", X_train_raw.isna().sum().sum())

    print("\nValidation rows with any NaN:", X_val_raw.isna().any(axis=1).sum())
    print("Validation total NaN cells:", X_val_raw.isna().sum().sum())

    # 3. Show the offending rows
    train_nan_rows = X_train_raw[X_train_raw.isna().any(axis=1)]
    val_nan_rows = X_val_raw[X_val_raw.isna().any(axis=1)]

    print("\nTrain rows containing NaNs:")
    print(train_nan_rows.head(20))

    print("\nValidation rows containing NaNs:")
    print(val_nan_rows.head(20))

In [ ]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]
class_names_mapping = {class_names[i]:i for i in range(len(class_names))}

def perform_decision_tree(X_train, X_val, y_train, y_val,
                          criterion, max_depth, min_samples_leaf, ccp_alpha):

    tree = DecisionTreeClassifier(
        criterion=criterion,              # "gini" or "entropy"
        max_depth=max_depth,              # None = grow until pure
        min_samples_leaf=min_samples_leaf,
        ccp_alpha=ccp_alpha,              # cost-complexity pruning strength
        class_weight="balanced",
        random_state=42,
    )
    tree.fit(X_train, y_train)
    predicted = tree.predict(X_val)  # is list of strings

    predicted_num = [class_names_mapping[x] for x in predicted]
    y_val_num = [class_names_mapping[x] for x in y_val]

    metrics = {
        "ARI": adjusted_rand_score(y_val_num, predicted_num),
        "NMI": normalized_mutual_info_score(y_val_num, predicted_num),
        "accuracy": accuracy_score(y_val_num, predicted_num),
        "macro_F1": f1_score(y_val, predicted, labels=class_names,
                             average="macro", zero_division=0),
    }

    return metrics, predicted

In [ ]:
from itertools import product
optimize_metric = "ARI"  # ARI, NMI, accuracy, macro_F1

use_features = features_C  # change feature set if desired

# Hyperparameter grid
criterion_values = ["gini", "entropy"]
max_depth_values = [2, 3, 4, 5, 6, 8, 10, None]
min_samples_leaf_values = [1, 2, 5, 10, 20]
ccp_alpha_values = [0.0, 0.0005, 0.001, 0.005, 0.01]

X_train_prepared = train[use_features]
X_val_prepared = val[use_features]
y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

metrics_df = []
val_preds = {}  # run_id -> predictions on validation set

grid = list(product(criterion_values, max_depth_values,
                    min_samples_leaf_values, ccp_alpha_values))
print(f"Searching {len(grid)} configurations...")

for run_id, (criterion, max_depth, min_leaf, ccp_alpha) in enumerate(grid):
    metrics, predicted = perform_decision_tree(
        X_train_prepared, X_val_prepared, y_train, y_val,
        criterion, max_depth, min_leaf, ccp_alpha
    )
    metrics["run_id"] = run_id
    metrics["criterion"] = criterion
    metrics["max_depth"] = max_depth
    metrics["min_samples_leaf"] = min_leaf
    metrics["ccp_alpha"] = ccp_alpha
    metrics_df.append(metrics)
    val_preds[run_id] = predicted

# Sort by the chosen metric; break ties in favour of the simpler tree
# (larger pruning strength, then larger leaves)
metrics_df = pd.DataFrame(metrics_df).sort_values(
    [optimize_metric, "ccp_alpha", "min_samples_leaf"],
    ascending=[False, False, False]
).reset_index(drop=True)
print("metrics df:")
print(metrics_df.head(15))

best = metrics_df.iloc[0]
print("\nBest hyperparameters:")
print(best[["criterion", "max_depth", "min_samples_leaf", "ccp_alpha", optimize_metric]])

val_predictions_best = val_preds[int(best["run_id"])]

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_val,                  # y_true first, so rows really are "Actual"
        val_predictions_best,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
)

print("\nValidation confusion matrix:")
print(confusion_df)

print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))